# Room-sweep transformer evaluation

Evaluates a `train_roomsweeps.py` checkpoint, i.e. the `TagPairTransformer` trained on room-sweep sub-clusters
from `roomsweeps_cluster_features.py`. It's the room-sweep version of `multitag_ml_training/ETF_eval_master.ipynb`.

1. Loads the checkpoint, its `_config.json` and the dataset it was trained on.
2. Rebuilds the **exact** train/val partition from `<run_stem>_split.csv` and applies the scalers stored in the checkpoint.
3. Reports overall and per-experiment metrics against two baselines:
   * **constant**: the per-ray mean of the training targets
   * **1-NN fingerprint**: the targets of the nearest training sample in the model's own (standardized) feature space
4. Plots predicted against true values, error against range, error by floor location, and predicted wall hits in the room.

> Every floor label always holds the same tag, so a random or experiment split re-measures the same links at the
> same places, and 1-NN lookup does well. Only a model that beats 1-NN on a `--val_columns` (location) holdout has
> learned something that transfers.

In [ ]:
import glob
import importlib
import json
import os
import sys

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Locate roomsweeps_ml/ so the CWD doesn't matter
HERE = os.path.abspath(".")
_cands = [HERE, os.path.join(HERE, "roomsweeps_ml"),
          os.path.join(HERE, "organized_multitag", "roomsweeps_processing", "roomsweeps_ml")]
HERE = next((d for d in _cands if os.path.isfile(os.path.join(d, "train_roomsweeps.py"))), None)
assert HERE, "launch from roomsweeps_ml/, roomsweeps_processing/ or the RFImaging root"
if HERE not in sys.path:
    sys.path.insert(0, HERE)

import train_roomsweeps as trs
importlib.reload(trs)
from train_roomsweeps import (SAMPLE_KEYS, SUB_KEYS, WALL_DIRS, load_for_args, load_split, preprocess,
                              scalers_from_checkpoint)
from MLScripts.models.transformer import TagPairTransformer   # path set up by train_roomsweeps

MARKERS_CSV = os.path.join(HERE, "..", "..", "roomscans", "merged_markers.csv")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
print("ml dir  :", HERE)
print("T2T_ROOT:", trs.T2T_ROOT)

## Configuration

`CHECKPOINT_PATH = None` picks the newest checkpoint in `SAVED_DIR`. `EVAL_MAX_GT_CM` sets which wall predictions
are scored. The loss clamps the ground truth at the training `--dist_clip`, so the model never learns distances
beyond it, and by default those rays are left out of the metrics. Set it to `np.inf` to score every ray.

In [ ]:
CHECKPOINT_PATH = None
SAVED_DIR       = os.path.join(HERE, "Saved")

EVAL_EXPERIMENTS       = None   # val experiments to report on; None → all
EVAL_TRAIN_SPLIT       = True   # also score the train split (overfitting gap)
EVAL_MAX_GT_CM         = None   # score rays whose GT ≤ this; None → the training dist_clip
NN_BASELINE            = True   # 1-NN fingerprint baseline (GPU if available)
NN_MAX_TRAIN           = 200_000  # subsample the train side of 1-NN beyond this
ROOM_LIMIT             = 300    # sub-clusters per experiment in the room plots

if CHECKPOINT_PATH is None:
    _c = sorted((p for p in glob.glob(os.path.join(SAVED_DIR, "ETF_rs_*.pth"))
                 if os.path.exists(p.replace(".pth", "_config.json"))), key=os.path.getmtime)
    if not _c:
        raise FileNotFoundError(f"No checkpoint in {SAVED_DIR}. Train one with train_roomsweeps.py")
    for _p in _c[-5:]:
        print(f"  {'->' if _p == _c[-1] else '  '} {os.path.basename(_p)}")
    CHECKPOINT_PATH = _c[-1]

with open(CHECKPOINT_PATH.replace(".pth", "_config.json")) as f:
    run_cfg = json.load(f)
train_args = run_cfg["args"]
if EVAL_MAX_GT_CM is None:
    EVAL_MAX_GT_CM = float(train_args.get("dist_clip", np.inf))
print("checkpoint :", os.path.basename(CHECKPOINT_PATH))
print("dataset    :", train_args["dataset"])
print("features   :", list(run_cfg["feature_slices"]),
      "(positions ON)" if train_args.get("include_positions") else "(no positions)")
print("split      :", {k: train_args.get(k) for k in ["val_experiments", "val_columns", "split", "no_purge"]})
print(f"scoring rays with GT ≤ {EVAL_MAX_GT_CM:g} cm")

## Load model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
ckpt  = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model = TagPairTransformer(**run_cfg["model_config"])
model.load_state_dict(ckpt["model_state_dict"])
model.to(device).eval()
print(f"Device {device}, {sum(p.numel() for p in model.parameters()):,} parameters, "
      f"best epoch {run_cfg['best_epoch']} / {train_args['epochs']}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(run_cfg["train_losses"], label="train")
ax.plot(run_cfg["val_losses"], label="val")
ax.axvline(run_cfg["best_epoch"] - 1, color="k", ls="--", alpha=0.5, label="best")
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("loss")
ax.set_title("Training curve"); ax.legend(); plt.tight_layout(); plt.show()

## Load data and the saved partition

In [ ]:
data = load_for_args(train_args)
assert data.X_raw.shape[2] == run_cfg["model_config"]["feat_dim"], "feat_dim mismatch: different feature flags?"
SPLIT_PATH = os.path.join(os.path.dirname(CHECKPOINT_PATH), run_cfg["split_file"])
train_idx, val_idx = load_split(SPLIT_PATH, data.meta_df)
split, _ = preprocess(data, train_idx, val_idx, scalers=scalers_from_checkpoint(ckpt))
assert (len(train_idx), len(val_idx)) == (run_cfg["n_train"], run_cfg["n_val"]), "split size differs from training"

meta_val_all = data.meta_df.iloc[val_idx].reset_index(drop=True)
meta_val_all.insert(0, "sample_idx", val_idx)
meta_train = data.meta_df.iloc[train_idx].reset_index(drop=True)
train_exps = sorted(meta_train["experiment"].unique().tolist())
val_exps   = sorted(meta_val_all["experiment"].unique().tolist())
exps = val_exps if EVAL_EXPERIMENTS is None else [e for e in EVAL_EXPERIMENTS if e in val_exps]
IS_UNSEEN = {e: e not in train_exps for e in exps}

mask_val   = meta_val_all["experiment"].isin(exps).to_numpy()
X_val_eval = split["X_val"][mask_val]
true_d     = split["Y_d_val"][mask_val]
true_a     = split["Y_a_val"][mask_val]
meta_val   = meta_val_all[mask_val].reset_index(drop=True)
vi         = val_idx[mask_val]                       # rows of `data` behind each val sample
col        = meta_val["experiment"].to_numpy()
true_d_tr, true_a_tr = split["Y_d_train"], split["Y_a_train"]
print(f"train {len(train_idx)} samples, experiments {train_exps}")
print(f"val   {len(meta_val)} samples, experiments {exps}  (unseen: {[e for e in exps if IS_UNSEEN[e]] or 'none'})")

## Inference and baselines

In [ ]:
@torch.no_grad()
def predict(X, batch=2048):
    outs = [model(torch.from_numpy(np.ascontiguousarray(X[i:i + batch])).to(device), None)
            for i in range(0, len(X), batch)]
    return (np.concatenate([o[0].cpu().numpy() for o in outs]),
            np.concatenate([o[1].cpu().numpy() for o in outs]))


@torch.no_grad()
def nearest_train(X_query, X_ref, batch=1024):
    """Index into X_ref of each query's nearest neighbour (Euclidean, flattened standardized features)."""
    R = torch.from_numpy(X_ref.reshape(len(X_ref), -1)).to(device)
    out = []
    for i in range(0, len(X_query), batch):
        q = torch.from_numpy(np.ascontiguousarray(X_query[i:i + batch]).reshape(-1, R.shape[1])).to(device)
        out.append(torch.cdist(q, R).argmin(1).cpu().numpy())
    return np.concatenate(out)


pred_d, pred_a = predict(X_val_eval)
DIST_CLIP = float(train_args.get("dist_clip", np.inf))
const_d = np.minimum(true_d_tr, DIST_CLIP).mean(0)     # what a no-input model would learn under the clipped loss
const_a = true_a_tr.mean(0)

if NN_BASELINE:
    ref = np.arange(len(train_idx))
    if len(ref) > NN_MAX_TRAIN:
        ref = np.sort(np.random.default_rng(0).choice(ref, NN_MAX_TRAIN, replace=False))
    nn_i = ref[nearest_train(X_val_eval, split["X_train"][ref])]
    nn_d, nn_a = true_d_tr[nn_i], true_a_tr[nn_i]
    meta_val["nn_experiment"] = meta_train["experiment"].to_numpy()[nn_i]
    meta_val["nn_same_location"] = (meta_train["labels"].to_numpy()[nn_i] == meta_val["labels"].to_numpy())
    print(f"1-NN: {meta_val['nn_same_location'].mean():.1%} of val samples' nearest train sample "
          f"is the same tags at the same floor labels")
else:
    nn_d = nn_a = None
print(f"pred_d {pred_d.shape}, GT range {true_d.min():.0f}–{true_d.max():.0f} cm, "
      f"{np.mean(true_d > EVAL_MAX_GT_CM):.1%} of rays beyond {EVAL_MAX_GT_CM:g} cm (not scored)")

## Metrics

Distances in **cm**, angles in **degrees**, over the rays with GT ≤ `EVAL_MAX_GT_CM`.

| metric | meaning |
|---|---|
| `mae_dist_cm`, `medae`, `p90` | mean / median / 90th-percentile absolute distance error |
| `acc<=10/20/50cm` | share of wall predictions within that tolerance |
| `const_cm` | MAE of the constant baseline (per-ray train mean) |
| `nn1_cm` | MAE of the 1-NN fingerprint baseline |
| `gain_vs_const`, `gain_vs_nn1` | baseline MAE − model MAE, so positive means the model is better |
| `mae_angle_deg` | wall incidence angle MAE |

Read `gain_vs_nn1` first. Raw MAE mostly reflects where the sub-clusters sit in the room.

In [ ]:
SCORED = true_d <= EVAL_MAX_GT_CM


def _mae(p, t, m):
    return float(np.nanmean(np.where(m, np.abs(p - t), np.nan))) if m.any() else np.nan


def metrics_for(sel):
    m = SCORED[sel]
    err = np.where(m, pred_d[sel] - true_d[sel], np.nan)
    a = np.abs(err)
    out = {
        "n":             int(sel.sum()),
        "mae_dist_cm":   float(np.nanmean(a)),
        "medae_dist_cm": float(np.nanmedian(a)),
        "p90_dist_cm":   float(np.nanpercentile(a, 90)),
        "bias_dist_cm":  float(np.nanmean(err)),
        "acc<=10cm":     float((a <= 10).sum() / m.sum()) if m.any() else np.nan,
        "acc<=20cm":     float((a <= 20).sum() / m.sum()) if m.any() else np.nan,
        "acc<=50cm":     float((a <= 50).sum() / m.sum()) if m.any() else np.nan,
        "const_cm":      _mae(np.broadcast_to(const_d, true_d[sel].shape), true_d[sel], m),
        "gt_dist_cm":    float(np.nanmean(np.where(m, true_d[sel], np.nan))),
        "mae_angle_deg": float(np.mean(np.abs(pred_a[sel] - true_a[sel]))),
    }
    out["gain_vs_const"] = out["const_cm"] - out["mae_dist_cm"]
    if nn_d is not None:
        out["nn1_cm"] = _mae(nn_d[sel], true_d[sel], m)
        out["gain_vs_nn1"] = out["nn1_cm"] - out["mae_dist_cm"]
    for j, w in enumerate(WALL_DIRS):
        out[f"mae_{w}_cm"] = _mae(pred_d[sel, j], true_d[sel, j], m[:, j])
    return out


def per_exp_table(order):
    rows = {e: metrics_for(col == e) for e in order}
    rows["ALL"] = metrics_for(np.isin(col, order))
    return pd.DataFrame(rows).T


exp_tbl = per_exp_table(exps)
exp_tbl["unseen"] = [IS_UNSEEN.get(e, np.nan) for e in exp_tbl.index]
_cols = [c for c in ["n", "unseen", "mae_dist_cm", "const_cm", "nn1_cm", "gain_vs_const", "gain_vs_nn1",
                     "medae_dist_cm", "p90_dist_cm", "bias_dist_cm", "acc<=10cm", "acc<=20cm", "acc<=50cm",
                     "gt_dist_cm", "mae_angle_deg"] if c in exp_tbl]
_fmt = {c: "{:.1f}" for c in _cols}
_fmt |= {c: "{:.1%}" for c in _cols if c.startswith("acc")}
_fmt |= {"n": "{:.0f}", "gain_vs_const": "{:+.1f}", "gain_vs_nn1": "{:+.1f}",
         "unseen": lambda v: "" if pd.isna(v) else ("YES" if v else "-")}
_sty = (exp_tbl[_cols].style.format(_fmt)
        .background_gradient(subset=["mae_dist_cm"], cmap="RdYlGn_r")
        .background_gradient(subset=[c for c in ["gain_vs_const", "gain_vs_nn1"] if c in _cols], cmap="RdYlGn")
        .set_caption("Val-split accuracy per experiment (unseen = experiment not in the train split)"))
display(_sty)

In [ ]:
# Model vs baselines, all scored val rays
_m = SCORED
summary = {"model": (pred_d, pred_a), "constant": (np.broadcast_to(const_d, true_d.shape),
                                                    np.broadcast_to(const_a, true_a.shape))}
if nn_d is not None:
    summary["1-NN fingerprint"] = (nn_d, nn_a)
display(pd.DataFrame({
    k: {"mae_dist_cm": _mae(d, true_d, _m),
        "medae_dist_cm": float(np.nanmedian(np.where(_m, np.abs(d - true_d), np.nan))),
        "acc<=20cm": float(((np.abs(d - true_d) <= 20) & _m).sum() / _m.sum()),
        "mae_angle_deg": float(np.mean(np.abs(a - true_a)))}
    for k, (d, a) in summary.items()}).T.style.format("{:.2f}").format({"acc<=20cm": "{:.1%}"})
    .set_caption("All val experiments"))

In [ ]:
# Per-ray MAE: experiments × {N, S, E, W}
wall_mat = exp_tbl.loc[exps, [f"mae_{w}_cm" for w in WALL_DIRS]].astype(float)
fig, ax = plt.subplots(figsize=(6, 0.4 * len(wall_mat) + 2))
im = ax.imshow(wall_mat.values, cmap="RdYlGn_r", aspect="auto")
ax.set_xticks(range(4), WALL_DIRS)
ax.set_yticks(range(len(wall_mat)), [f"exp {e}" for e in wall_mat.index])
for i in range(wall_mat.shape[0]):
    for j in range(4):
        ax.text(j, i, f"{wall_mat.values[i, j]:.0f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax, label="MAE [cm]")
ax.set_title("Per-ray distance MAE by experiment"); plt.tight_layout(); plt.show()

### Ranking chart

In [ ]:
plot_tbl = exp_tbl.drop(index="ALL").sort_values("mae_dist_cm")
labels_ = [f"exp {e}" + (" *" if IS_UNSEEN[e] else "") for e in plot_tbl.index]
fig, axes = plt.subplots(1, 3, figsize=(16, 0.32 * len(plot_tbl) + 3))
ax = axes[0]
ax.barh(labels_, plot_tbl["mae_dist_cm"], color="steelblue", label="model")
ax.scatter(plot_tbl["const_cm"], labels_, marker="|", s=200, color="k", label="constant", zorder=5)
if "nn1_cm" in plot_tbl:
    ax.scatter(plot_tbl["nn1_cm"], labels_, marker="x", s=40, color="darkorange", label="1-NN", zorder=5)
ax.axvline(exp_tbl.loc["ALL", "mae_dist_cm"], color="red", ls="--", label=f"all ({exp_tbl.loc['ALL', 'mae_dist_cm']:.1f} cm)")
ax.set_xlabel("MAE [cm]"); ax.set_title("Wall-distance error (* = unseen experiment)"); ax.legend(fontsize=8)
ax = axes[1]
y_ = np.arange(len(plot_tbl)); w_ = 0.38
ax.barh(y_ - w_ / 2, plot_tbl["acc<=20cm"] * 100, height=w_, label="≤ 20 cm", color="seagreen")
ax.barh(y_ + w_ / 2, plot_tbl["acc<=50cm"] * 100, height=w_, label="≤ 50 cm", color="darkseagreen")
ax.set_yticks(y_, labels_); ax.set_xlabel("% of wall predictions"); ax.set_title("Within tolerance"); ax.legend(fontsize=8)
ax = axes[2]
ax.barh(labels_, plot_tbl["mae_angle_deg"], color="indianred")
ax.set_xlabel("MAE [deg]"); ax.set_title("Wall-angle error")
plt.tight_layout(); plt.show()

### Prediction vs ground truth, per experiment

Grey band: GT beyond `EVAL_MAX_GT_CM` (not scored, and beyond what the clipped loss trains for).

In [ ]:
lim = float(np.ceil(min(true_d.max(), 1.3 * EVAL_MAX_GT_CM) / 50) * 50)
ncol = 4
nrow = int(np.ceil((len(exps) + 1) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.9 * nrow), squeeze=False)
for k, e in enumerate(exps + ["ALL"]):
    ax = axes.flat[k]
    sel = np.ones(len(col), bool) if e == "ALL" else col == e
    for j, w in enumerate(WALL_DIRS):
        ax.scatter(true_d[sel, j], np.clip(pred_d[sel, j], 0, lim), s=2, alpha=0.25, label=w)
    ax.plot([0, lim], [0, lim], "r--", lw=1)
    if EVAL_MAX_GT_CM < lim:
        ax.axvspan(EVAL_MAX_GT_CM, lim, color="0.85", zorder=0)
    ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.set_aspect("equal")
    ax.set_title(f"{'all' if e == 'ALL' else 'exp ' + str(e)}  MAE {exp_tbl.loc[e, 'mae_dist_cm']:.1f} cm", fontsize=10)
    if k % ncol == 0:
        ax.set_ylabel("pred [cm]")
    ax.set_xlabel("GT [cm]")
axes.flat[0].legend(fontsize=7, markerscale=4)
for ax in axes.flat[len(exps) + 1:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(true_a, pred_a, s=2, alpha=0.15, color="indianred")
ax.plot([-90, 90], [-90, 90], "r--", lw=1); ax.set_xlim(-90, 90); ax.set_ylim(-90, 90); ax.set_aspect("equal")
ax.set_xlabel("GT angle [deg]"); ax.set_ylabel("pred angle [deg]")
ax.set_title(f"Wall angle, all val (MAE {exp_tbl.loc['ALL', 'mae_angle_deg']:.1f}°)"); plt.tight_layout(); plt.show()

### Error distributions: model vs baselines

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
ax = axes[0]
for name, (d, _) in summary.items():
    err = np.sort(np.abs(d - true_d)[SCORED])
    ax.plot(err, np.arange(1, len(err) + 1) / len(err), lw=1.6, label=name)
ax.set_xlim(0, 200); ax.set_ylim(0, 1); ax.grid(alpha=0.3)
ax.set_xlabel("|distance error| [cm]"); ax.set_ylabel("CDF"); ax.set_title("Error CDF, all val"); ax.legend()

ax = axes[1]
for e in exps:
    m = SCORED[col == e]
    err = np.sort(np.abs(pred_d[col == e] - true_d[col == e])[m])
    ax.plot(err, np.arange(1, len(err) + 1) / max(len(err), 1), lw=1, label=f"exp {e}")
ax.set_xlim(0, 200); ax.set_ylim(0, 1); ax.grid(alpha=0.3)
ax.set_xlabel("|distance error| [cm]"); ax.set_title("Model error CDF per experiment"); ax.legend(fontsize=6, ncol=2)

ax = axes[2]
edges = np.arange(0, min(true_d.max(), EVAL_MAX_GT_CM) + 25, 25)
cent = 0.5 * (edges[:-1] + edges[1:])
b = np.digitize(true_d, edges) - 1
for name, (d, _) in summary.items():
    e_ = np.abs(d - true_d)
    ax.plot(cent, [e_[(b == i) & SCORED].mean() if ((b == i) & SCORED).sum() >= 20 else np.nan
                   for i in range(len(cent))], marker="o", ms=3, label=name)
ax.set_xlabel("GT wall distance [cm]"); ax.set_ylabel("mean |error| [cm]")
ax.set_title("Error vs range (bins < 20 rays omitted)"); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## The room

Everything is in the merged-scan frame shared by all experiments. Lines are the four walls from `find_walls()`,
and dots are the floor-label markers. The first figure colours each val sub-cluster's centroid by its model MAE
and shows where on the floor the model works. The second draws the predicted wall hits
(centroid + predicted distance × ray direction) per experiment. If the model is right, they land on the walls.

In [ ]:
room = data.room
WP, WN = np.asarray(room["wall_points"]), np.asarray(room["wall_normals"])


def _corner(i, j):
    A = np.stack([WN[i], WN[j]])
    return np.linalg.solve(A, [WN[i] @ WP[i], WN[j] @ WP[j]])


_names = room["wall_names"]
_ix = {n: i for i, n in enumerate(_names)}
_ring = [_ix["Edge02 wall"], _ix["Row-1 wall"], _ix["Edge01 wall"], _ix["Row-8 wall"]]
ROOM_POLY = np.array([_corner(_ring[k], _ring[(k + 1) % 4]) for k in range(4)] + [_corner(_ring[0], _ring[1])])
markers = pd.read_csv(MARKERS_CSV).set_index("index").sort_index() if os.path.isfile(MARKERS_CSV) else None

cmap20 = plt.get_cmap("tab20")
ALL_EXPS = sorted(data.meta_df["experiment"].unique().tolist())
EXP_COLORS = {e: cmap20(i % 20) for i, e in enumerate(ALL_EXPS)}
WALL_COLORS = dict(zip(WALL_DIRS, ["#2a78d6", "#eb6834", "#1baf7a", "#6250d6"]))


def draw_room(ax):
    ax.plot(ROOM_POLY[:, 0], ROOM_POLY[:, 1], color="0.3", lw=2, zorder=1)
    if markers is not None:
        ax.scatter(markers["x"], markers["y"], s=4, c="0.75", zorder=1)
    ax.set_aspect("equal"); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.grid(alpha=0.2)


# Model MAE per sub-cluster (mean over its runs / orderings) at its centroid
_err = np.where(SCORED, np.abs(pred_d - true_d), np.nan)
loc = (meta_val.assign(err=np.nanmean(_err, axis=1))
       .groupby(["experiment", "combination_id"], as_index=False)
       .agg(x=("centroid_x", "first"), y=("centroid_y", "first"), err=("err", "mean")))
fig, ax = plt.subplots(figsize=(9, 8))
draw_room(ax)
sc = ax.scatter(loc["x"], loc["y"], c=loc["err"], cmap="RdYlGn_r", s=14, vmin=0,
                vmax=np.nanpercentile(loc["err"], 95), zorder=3)
plt.colorbar(sc, ax=ax, label="model MAE [cm], scored rays", shrink=0.8)
ax.set_title("Val sub-cluster centroids coloured by error"); plt.tight_layout(); plt.show()

In [ ]:
rng = np.random.default_rng(0)
first = ~meta_val.duplicated(SUB_KEYS[:2]).to_numpy()             # one sample per combination
dirs_val = data.ray_dirs[vi][:, :, :2]
cent_val = data.centroids[vi][:, :2]
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, (title, D) in zip(axes, [("predicted wall hits, by experiment", pred_d),
                                 ("predicted wall hits, by ray", pred_d)]):
    draw_room(ax)
    for e in exps:
        cand = np.flatnonzero((col == e) & first)
        pick = rng.choice(cand, min(ROOM_LIMIT, len(cand)), replace=False)
        hits = cent_val[pick, None] + np.clip(D[pick], 0, None)[..., None] / 100 * dirs_val[pick]
        if ax is axes[0]:
            ax.scatter(hits[..., 0].ravel(), hits[..., 1].ravel(), s=3, alpha=0.5, color=EXP_COLORS[e], zorder=2)
        else:
            for j, w in enumerate(WALL_DIRS):
                ax.scatter(hits[:, j, 0], hits[:, j, 1], s=3, alpha=0.4, color=WALL_COLORS[w], zorder=2)
        ax.scatter(cent_val[pick, 0], cent_val[pick, 1], s=4, c="k", zorder=3)
    ax.set_title(title)
pad = 1.0
for ax in axes:
    ax.set_xlim(ROOM_POLY[:, 0].min() - pad, ROOM_POLY[:, 0].max() + pad)
    ax.set_ylim(ROOM_POLY[:, 1].min() - pad, ROOM_POLY[:, 1].max() + pad)
axes[0].legend(handles=[Line2D([], [], ls="", marker="o", color=EXP_COLORS[e], label=f"exp {e}") for e in exps],
               fontsize=7, ncol=2, loc="upper left")
axes[1].legend(handles=[Line2D([], [], ls="", marker="o", color=c, label=w) for w, c in WALL_COLORS.items()],
               fontsize=8)
plt.suptitle("Black: sub-cluster centroids (ray origins). Grey line: walls.", y=0.99)
plt.tight_layout(); plt.show()

### One sub-cluster in detail

Tags (numbered by permutation position, 0 sets the +X axis in the local frame), its four rays to the ground-truth
hits (thin) and the predicted hits (×). Change `EXAMPLE` to browse.

In [ ]:
EXAMPLE = int(np.argsort(np.nanmean(_err, axis=1))[len(_err) // 2])   # a median-error sample
r = vi[EXAMPLE]
fig, ax = plt.subplots(figsize=(8, 7))
draw_room(ax)
c0 = data.centroids[r, :2]
xyz = np.array([[markers.loc[l, "x"], markers.loc[l, "y"]] for l in data.labels[r]]) if markers is not None else None
if xyz is not None:
    ax.scatter(xyz[:, 0], xyz[:, 1], s=40, c="k", zorder=4)
    for k, (x_, y_) in enumerate(xyz):
        ax.annotate(str(k), (x_, y_), textcoords="offset points", xytext=(4, 4), fontsize=9)
for j, w in enumerate(WALL_DIRS):
    d = data.ray_dirs[r, j, :2]
    gt = c0 + true_d[EXAMPLE, j] / 100 * d
    pr = c0 + max(pred_d[EXAMPLE, j], 0) / 100 * d
    ax.plot([c0[0], gt[0]], [c0[1], gt[1]], color=WALL_COLORS[w], lw=1, label=f"{w}: GT {true_d[EXAMPLE, j]:.0f}, "
                                                                            f"pred {pred_d[EXAMPLE, j]:.0f} cm")
    ax.scatter(*pr, marker="x", s=80, color=WALL_COLORS[w], zorder=5)
ax.legend(fontsize=8)
m_ = meta_val.iloc[EXAMPLE]
ax.set_title(f"exp {m_.experiment}, tags {m_.tags}, labels {m_.labels}, run {m_.run}")
plt.tight_layout(); plt.show()

## Train-split accuracy

For experiments on both sides, the train↔val gap is the overfitting check. Set `EVAL_TRAIN_SPLIT = False` to skip.

In [ ]:
if EVAL_TRAIN_SPLIT:
    pred_d_tr, pred_a_tr = predict(split["X_train"])
    m_tr = true_d_tr <= EVAL_MAX_GT_CM
    col_tr = meta_train["experiment"].to_numpy()
    train_tbl = pd.DataFrame({
        e: {"n": int((col_tr == e).sum()),
            "mae_train_cm": _mae(pred_d_tr[col_tr == e], true_d_tr[col_tr == e], m_tr[col_tr == e]),
            "mae_val_cm": exp_tbl.loc[e, "mae_dist_cm"] if e in exp_tbl.index else np.nan}
        for e in train_exps}).T
    train_tbl.loc["ALL"] = [len(col_tr), _mae(pred_d_tr, true_d_tr, m_tr), exp_tbl.loc["ALL", "mae_dist_cm"]]
    train_tbl["gap_cm"] = train_tbl["mae_val_cm"] - train_tbl["mae_train_cm"]
    display(train_tbl.style.format({"n": "{:.0f}", "mae_train_cm": "{:.1f}", "mae_val_cm": "{:.1f}",
                                    "gap_cm": "{:+.1f}"}, na_rep="")
            .set_caption("Train-split MAE per experiment (mae_val_cm: same experiment in val, if any)"))
else:
    train_tbl = None
    print("skipped")

## Export

In [ ]:
stem = os.path.join(os.path.dirname(CHECKPOINT_PATH), run_cfg["run_stem"])
exp_tbl.rename_axis("experiment").to_csv(stem + "_perexp_eval.csv")
_p = meta_val[["sample_idx"] + SAMPLE_KEYS + ["tags", "labels", "centroid_x", "centroid_y"]].copy()
for j, w in enumerate(WALL_DIRS):
    _p[f"gt_dist_{w}_cm"], _p[f"pred_dist_{w}_cm"] = true_d[:, j], pred_d[:, j]
    if nn_d is not None:
        _p[f"nn1_dist_{w}_cm"] = nn_d[:, j]
    _p[f"gt_angle_{w}_deg"], _p[f"pred_angle_{w}_deg"] = true_a[:, j], pred_a[:, j]
_p.to_csv(stem + "_val_predictions.csv", index=False)
print("Per-experiment metrics →", stem + "_perexp_eval.csv")
print("Val predictions        →", stem + "_val_predictions.csv")